In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd()

# If notebook is opened from Data/scripts, this points to Data/
DATA_DIR = BASE_DIR.parent

RAW_PATH = DATA_DIR / "raw" / "tabular" / "australian_postcodes.csv"
OUT_DIR = DATA_DIR / "processed" / "postcode_lookup"
OUT_PATH = OUT_DIR / "postcode_locality_lookup_vic.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Reading:", RAW_PATH)
df = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()


Reading: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\raw\tabular\australian_postcodes.csv
Rows: 18559
Columns: ['id', 'postcode', 'locality', 'state', 'long', 'lat', 'dc', 'type', 'status', 'sa3', 'sa3name', 'sa4', 'sa4name', 'region', 'Lat_precise', 'Long_precise', 'SA1_CODE_2021', 'SA1_NAME_2021', 'SA2_CODE_2021', 'SA2_NAME_2021', 'SA3_CODE_2021', 'SA3_NAME_2021', 'SA4_CODE_2021', 'SA4_NAME_2021', 'RA_2011', 'RA_2016', 'RA_2021', 'RA_2021_NAME', 'MMM_2015', 'MMM_2019', 'ced', 'altitude', 'chargezone', 'phn_code', 'phn_name', 'lgaregion', 'lgacode', 'electorate', 'electoraterating', 'sed_code', 'sed_name']


,id,postcode,locality,state,long,lat,dc,type,status,sa3,...,altitude,chargezone,phn_code,phn_name,lgaregion,lgacode,electorate,electoraterating,sed_code,sed_name
0,230,0200,ANU,ACT,149.119,-35.2777,,,Updated 3-Dec-2022,,...,,N2,,,Unincorporated ACT,89399,Durack,,,
1,21820,0200,Australian National University,ACT,149.1189,-35.2777,,,Updated 3-Dec-2022,,...,,N2,,,Unincorporated ACT,89399,Durack,,,
2,232,0800,DARWIN,NT,130.83668,-12.458684,,,Updated 3-Dec-2022,70101,...,,NT1,PHN701,Northern Territory,Darwin Waterfront Precinct,71150,Solomon,Inner Metropolitan,70022,Port Darwin
3,24049,0800,DARWIN CITY,NT,130.83668,-12.458684,,,Updated 3-Dec-2022,70101,...,,NT1,PHN701,Northern Territory,Darwin Waterfront Precinct,71150,Solomon,Inner Metropolitan,70022,Port Darwin
4,233,0801,DARWIN,NT,130.83668,-12.458684,,,Updated 3-Dec-2022,70101,...,,NT1,PHN701,,Darwin,71000,Lingiari,Rural,,


In [2]:
# Keep Victoria only
vic = df[df["state"].str.upper().eq("VIC")].copy()

# Normalise postcode to 4 digits
vic["postcode"] = (
    vic["postcode"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(4)
)

# Clean locality text
vic["locality"] = (
    vic["locality"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

# Prefer precise coordinates, fall back to long/lat
vic["lat_clean"] = pd.to_numeric(
    vic["Lat_precise"].where(vic["Lat_precise"].ne(""), vic["lat"]),
    errors="coerce"
)

vic["lng_clean"] = pd.to_numeric(
    vic["Long_precise"].where(vic["Long_precise"].ne(""), vic["long"]),
    errors="coerce"
)

# Normalise type/status
vic["type_clean"] = vic["type"].astype(str).str.strip()
vic["status_clean"] = vic["status"].astype(str).str.strip()

# Keep rows useful for website postcode search
clean = vic[
    vic["postcode"].str.fullmatch(r"\d{4}") &
    vic["locality"].ne("") &
    vic["lat_clean"].notna() &
    vic["lng_clean"].notna() &
    ~vic["type_clean"].str.lower().eq("post office boxes")
].copy()

print("VIC rows:", len(vic))
print("Clean rows:", len(clean))
print("Unique postcodes:", clean["postcode"].nunique())
print("Unique localities:", clean["locality"].nunique())

clean.head()


VIC rows: 3539
Clean rows: 3511
Unique postcodes: 724
Unique localities: 3338


,id,postcode,locality,state,long,lat,dc,type,status,sa3,...,lgaregion,lgacode,electorate,electoraterating,sed_code,sed_name,lat_clean,lng_clean,type_clean,status_clean
6207,4746,3000,MELBOURNE,VIC,144.9825846,-37.81443733,CITY DELIVERY CENTRE,Delivery Area,Updated 17-Mar-2024 AUSPOST,20604,...,Melbourne,24600,Melbourne,Inner Metropolitan,24703,Melbourne (Northern Metropolitan),-37.815207,144.963937,Delivery Area,Updated 17-Mar-2024 AUSPOST
6209,4748,3002,EAST MELBOURNE,VIC,144.9825846,-37.81443733,CITY DELIVERY CENTRE,Delivery Area,Updated 17-Mar-2024 AUSPOST,20604,...,Melbourne,24600,Melbourne,Inner Metropolitan,24703,Melbourne (Northern Metropolitan),-37.816144,144.980459,Delivery Area,Updated 17-Mar-2024 AUSPOST
6210,4749,3003,WEST MELBOURNE,VIC,144.949592,-37.810871,CITY DELIVERY CENTRE,Delivery Area,Updated 6-Feb-2020,20604,...,Melbourne,24600,Melbourne,Inner Metropolitan,24703,Melbourne (Northern Metropolitan),-37.811450,144.925397,Delivery Area,Updated 6-Feb-2020
6211,4750,3004,MELBOURNE,VIC,144.9825846,-37.81443733,CITY DELIVERY CENTRE,Delivery Area,Updated 17-Mar-2024 AUSPOST,20605,...,Melbourne,24600,Macnamara,Inner Metropolitan,26906,Prahran (Southern Metropolitan),-37.830158,144.980459,Delivery Area,Updated 17-Mar-2024 AUSPOST
6212,4751,3004,ST KILDA ROAD CENTRAL,VIC,144.970161,-37.844246,Melbourne Metro,Delivery Area,Updated 6-Feb-2020,20605,...,Melbourne,24600,Macnamara,Inner Metropolitan,26906,Prahran (Southern Metropolitan),-37.836800,144.976000,Delivery Area,Updated 6-Feb-2020


In [8]:
output = pd.DataFrame({
    "postcode": clean["postcode"],
    "locality": clean["locality"],
    "state": clean["state"].str.upper(),
    "type": clean["type_clean"],
    "status": clean["status_clean"],
    "lat": clean["lat_clean"].round(7),
    "lng": clean["lng_clean"].round(7),
    "sa1_code_2021": clean.get("SA1_CODE_2021", ""),
    "sa1_name_2021": clean.get("SA1_NAME_2021", ""),
    "sa2_code_2021": clean.get("SA2_CODE_2021", ""),
    "sa2_name_2021": clean.get("SA2_NAME_2021", ""),
    "lga_name": clean.get("lgaregion", ""),
    "lga_code": clean.get("lgacode", ""),
})

output = output.drop_duplicates(
    subset=["postcode", "locality", "lat", "lng"]
).sort_values(["postcode", "locality"])

print("Output rows:", len(output))
output.head()


Output rows: 3511


,postcode,locality,state,type,status,lat,lng,sa1_code_2021,sa1_name_2021,sa2_code_2021,sa2_name_2021,lga_name,lga_code
6207,3000,MELBOURNE,VIC,Delivery Area,Updated 17-Mar-2024 AUSPOST,-37.815207,144.963937,20604150622,North Melbourne,206041506,North Melbourne,Melbourne,24600
6209,3002,EAST MELBOURNE,VIC,Delivery Area,Updated 17-Mar-2024 AUSPOST,-37.816144,144.980459,20604111915,East Melbourne,206041119,East Melbourne,Melbourne,24600
6210,3003,WEST MELBOURNE,VIC,Delivery Area,Updated 6-Feb-2020,-37.811450,144.925397,20604151016,West Melbourne,206041510,West Melbourne - Residential,Melbourne,24600
6211,3004,MELBOURNE,VIC,Delivery Area,Updated 17-Mar-2024 AUSPOST,-37.830158,144.980459,20605151429,Melbourne,206051514,St Kilda - West,Melbourne,24600
6212,3004,ST KILDA ROAD CENTRAL,VIC,Delivery Area,Updated 6-Feb-2020,-37.836800,144.976000,20605151429,Melbourne,206051514,St Kilda - West,Melbourne,24600


In [9]:
# Quick quality checks

summary = {
    "rows": len(output),
    "unique_postcodes": output["postcode"].nunique(),
    "unique_localities": output["locality"].nunique(),
    "missing_sa2_code": int(output["sa2_code_2021"].eq("").sum()),
    "missing_lga_name": int(output["lga_name"].eq("").sum()),
    "min_postcode": output["postcode"].min(),
    "max_postcode": output["postcode"].max(),
}

summary



{'rows': 3511,
 'unique_postcodes': 724,
 'unique_localities': 3338,
 'missing_sa2_code': 32,
 'missing_lga_name': 22,
 'min_postcode': '3000',
 'max_postcode': '9999'}

In [10]:
# Show postcodes with many localities.
# This is expected in Australia, and confirms why postcode search should return grouped/multiple results.

postcode_counts = (
    output.groupby("postcode")["locality"]
    .nunique()
    .reset_index(name="locality_count")
    .sort_values("locality_count", ascending=False)
)

postcode_counts.head(20)


,postcode,locality_count
289,3352,62
320,3401,47
407,3551,41
494,3691,37
487,3678,35
621,3875,35
366,3478,34
594,3825,31
428,3579,30
365,3477,30


In [11]:
# Save cleaned file

output.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
print("Size MB:", round(OUT_PATH.stat().st_size / 1024 / 1024, 2))


Saved: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\postcode_lookup\postcode_locality_lookup_vic.csv
Size MB: 0.47


In [12]:
output.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
print("Size MB:", round(OUT_PATH.stat().st_size / 1024 / 1024, 2))


Saved: C:\Users\AmanRoy\Documents\GitHub\NeighbourWise\Data\processed\postcode_lookup\postcode_locality_lookup_vic.csv
Size MB: 0.47
